# H20a: Cosine Compounding -- Does a +0.004/step Advantage Become 19x?

## The Central Paradox This Experiment Resolves

Two prior experiments in this research program produced results that, taken at face value, appear contradictory:

- **H15a (Direction Quality Metric):** Muon's per-step alignment with the Newton direction is only +0.004 better than NormSGD. That is, at any single step, Muon's update direction has a cosine similarity to the true Newton step that exceeds NormSGD's by a margin so small it seems negligible.

- **H3 (Normalized SGD vs Muon):** Muon beats NormSGD by ~19x on final loss. The cumulative outcome is enormous -- not the kind of gap one expects from a +0.004 edge per step.

**How does a near-invisible per-step advantage produce a massive final-loss gap?**

## Hypothesis: Geometric Compounding of Directional Quality

The proposed mechanism is **compounding via curvature-landscape coupling**. The argument runs as follows:

1. At step $t$, Muon's update direction is slightly closer to the Newton direction than NormSGD's.
2. Because the Newton direction accounts for local curvature, being closer to it means Muon lands in a region of parameter space where the loss surface is slightly more favorable -- gradients are slightly more informative, the Hessian is slightly better conditioned.
3. At step $t+1$, Muon starts from this slightly better point. Its gradient is slightly more useful, and its Newton-Schulz orthogonalization extracts slightly more curvature information from it.
4. This compounds multiplicatively: each step's advantage feeds the next step's starting conditions.

If this is correct, the loss ratio should grow as:

$$\text{loss\_ratio}(T) \;\sim\; \exp\!\Big(\alpha \cdot \sum_{t=1}^{T} \big[\cos(\theta_{\text{Muon}}^{(t)}, d_N^{(t)}) - \cos(\theta_{\text{NormSGD}}^{(t)}, d_N^{(t)})\big]\Big)$$

That is, $\log(\text{loss\_ratio})$ should be **linear in the cumulative cosine advantage**, not merely in the step count.

## Experimental Design

**Protocol:** Train both Muon and NormSGD from the same initialization for 500 steps on a 4-layer deep linear network (32x32). At every 50 steps, compute:

1. The approximate Newton direction at each optimizer's current point (via conjugate-gradient inversion of the Hessian)
2. $\cos(\text{optimizer\_step}, \text{Newton\_direction})$ for each optimizer
3. The running cumulative cosine advantage: $\sum [\cos_{\text{Muon}} - \cos_{\text{NormSGD}}]$
4. The loss ratio: $\text{loss}_{\text{NormSGD}} / \text{loss}_{\text{Muon}}$

**Key Test:** Fit two regression models:
- **(A)** $\log(\text{loss\_ratio})$ vs. cumulative cosine advantage -- if linear, this confirms geometric compounding through directional quality
- **(B)** $\log(\text{loss\_ratio})$ vs. step count $T$ -- if linear, this is a simpler exponential divergence that does not require the cosine mechanism

Compare $R^2$ of both fits. If $R^2_A > R^2_B$, the cosine advantage is the **causal driver**, not just a correlate of time.

**Scale:** 4-layer, 32x32, 500 steps, 5 seeds. The CG-Newton approximation makes Hessian computation tractable at this scale (4096 parameters).

In [ ]:
import numpy as np
import os
print("NumPy version:", np.__version__)
print("Environment ready.")

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
print("Script directory:", SCRIPT_DIR)

## Hyperparameters and Measurement Schedule

**Network:** A 4-layer deep linear network with 32x32 weight matrices. This gives $4 \times 32^2 = 4096$ total parameters -- large enough for meaningful curvature structure, small enough that conjugate-gradient Newton is tractable.

**Training:** 500 steps with momentum 0.9. Both optimizers use the same momentum coefficient to ensure the only difference is the per-layer update rule (Newton-Schulz orthogonalization vs. Frobenius normalization).

**Measurement schedule:** We compute Newton directions every 50 steps (10 checkpoints total). At each checkpoint, we use CG-Newton with 30 iterations of conjugate gradient, which approximates $H^{-1}g$ without ever forming the full $4096 \times 4096$ Hessian.

**Newton-Schulz iterations:** 5 iterations for the Muon orthogonalization, matching the standard Muon implementation.

In [ ]:
DIM = 32
N_LAYERS = 4
NUM_STEPS = 500
NUM_SEEDS = 5
BATCH_SIZE = 64
MOMENTUM = 0.9
NS_ITERS = 5

print("=== Hyperparameters ===")
print(f"  Network: {N_LAYERS}-layer deep linear, {DIM}x{DIM} per layer")
print(f"  Total parameters: {N_LAYERS * DIM * DIM}")
print(f"  Training steps: {NUM_STEPS}")
print(f"  Seeds: {NUM_SEEDS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Momentum: {MOMENTUM}")
print(f"  Newton-Schulz iterations: {NS_ITERS}")
print(f"  Hessian size (if full): {N_LAYERS * DIM * DIM}x{N_LAYERS * DIM * DIM} = {(N_LAYERS * DIM * DIM)**2:,} entries")
print(f"  -> Using CG-Newton instead of full Hessian inversion")

In [ ]:
# Checkpoints where we compute Hessian-based Newton direction
# We measure every 50 steps for the cosine computation
MEASURE_EVERY = 50
CHECKPOINTS = list(range(MEASURE_EVERY, NUM_STEPS + 1, MEASURE_EVERY))

print(f"Measurement checkpoints ({len(CHECKPOINTS)} total): {CHECKPOINTS}")
print(f"  Each checkpoint requires 2 CG-Newton solves (one per optimizer)")
print(f"  Each CG solve uses up to 30 Hessian-vector products")
print(f"  Each Hessian-vector product = 2 forward-backward passes")
print(f"  Total expensive operations per seed: ~{len(CHECKPOINTS) * 2 * 30 * 2} gradient evaluations for Newton")

## Network Architecture and Core Computations

### Newton-Schulz Orthogonalization (Muon's Core)

The Newton-Schulz iteration approximates the polar factor $U$ of a matrix $M = U \Sigma V^T$ by iteratively computing $X_{k+1} = \frac{3}{2}X_k - \frac{1}{2}X_k(X_k^T X_k)$. After convergence, $X \approx UV^T$, the nearest orthogonal matrix. This is the operation that gives Muon its curvature-awareness: by projecting each layer's gradient onto the orthogonal group, Muon equalizes all singular values of the update, effectively treating all directions of the loss landscape with equal weight regardless of their curvature.

The key insight: this equalization is precisely what makes Muon approximate a Newton step. In the eigenbasis of the Hessian, Newton rescales each gradient component by the inverse eigenvalue. Muon's orthogonalization achieves a crude version of this by making all components equal-magnitude, which is optimal when curvature eigenvalues are spread across orders of magnitude.

In [ ]:
def newton_schulz(M, n_iters=NS_ITERS):
    norm = np.linalg.norm(M, 'fro')
    if norm < 1e-15:
        return M
    X = M / norm
    for _ in range(n_iters):
        A = X.T @ X
        X = 1.5 * X - 0.5 * X @ A
    return X

### Weight Initialization and Forward Pass

Weights are initialized as $W_l = I + 0.1 \cdot \mathcal{N}(0,1)$, i.e., near-identity with small random perturbation. This ensures the initial product $W_L \cdots W_1 \approx I + \epsilon$, so the network starts near the identity map. The perturbation scale 0.1 is small enough that training begins in a well-conditioned region but large enough that the gradient landscape has nontrivial curvature structure.

The forward pass computes $Y = W_4 W_3 W_2 W_1 X$, and the loss is $\frac{1}{2} \| Y_{\text{pred}} - Y_{\text{target}} \|_F^2$.

In [ ]:
def init_weights(seed):
    rng = np.random.RandomState(seed)
    return [np.eye(DIM) + rng.randn(DIM, DIM) * 0.1 for _ in range(N_LAYERS)]

In [ ]:
def forward(weights, X):
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

In [ ]:
def compute_loss(weights, X, Y):
    pred = forward(weights, X)
    return 0.5 * np.mean(np.sum((pred - Y) ** 2, axis=0))

### Gradient Computation and Parameter Packing

Gradients are computed via manual backpropagation through the linear chain. For a deep linear network $f(x) = W_L \cdots W_1 x$, the gradient with respect to layer $l$ is:

$$\nabla_{W_l} \mathcal{L} = \delta_l \cdot a_{l-1}^T$$

where $\delta_l$ is the error signal backpropagated through layers $L, L-1, \ldots, l+1$ and $a_{l-1}$ is the activation entering layer $l$.

The `pack` and `unpack` functions flatten all layer weights into a single vector $\theta \in \mathbb{R}^{4096}$. This is essential for the CG-Newton computation, which operates in the full parameter space.

In [ ]:
def compute_gradients(weights, X, Y):
    L = len(weights)
    N = X.shape[1]
    acts = [X.copy()]
    for W in weights:
        acts.append(W @ acts[-1])
    delta = (acts[-1] - Y) / N
    grads = [None] * L
    for l in range(L - 1, -1, -1):
        grads[l] = delta @ acts[l].T
        if l > 0:
            delta = weights[l].T @ delta
    return grads

In [ ]:
def pack(weights):
    return np.concatenate([W.ravel() for W in weights])

In [ ]:
def unpack(vec):
    ws = []
    offset = 0
    for _ in range(N_LAYERS):
        ws.append(vec[offset:offset + DIM * DIM].reshape(DIM, DIM))
        offset += DIM * DIM
    return ws

In [ ]:
def make_data(seed):
    rng = np.random.RandomState(seed)
    W_target = rng.randn(DIM, DIM) * 0.5
    X = rng.randn(DIM, BATCH_SIZE) * 0.3
    Y = W_target @ X
    return X, Y

In [ ]:
def loss_fn_flat(theta, X, Y):
    ws = unpack(theta)
    return compute_loss(ws, X, Y)

In [ ]:
def grad_fn_flat(theta, X, Y):
    ws = unpack(theta)
    grads = compute_gradients(ws, X, Y)
    return pack(grads), grads

In [ ]:
def hessian_fd(theta, X, Y, eps=1e-5):
    """Finite-difference Hessian. Only used on subsampled params for 32x32."""
    n = len(theta)
    H = np.zeros((n, n))
    g0, _ = grad_fn_flat(theta, X, Y)
    for i in range(n):
        tp = theta.copy()
        tp[i] += eps
        gp, _ = grad_fn_flat(tp, X, Y)
        H[:, i] = (gp - g0) / eps
    return 0.5 * (H + H.T)

In [ ]:
def cosine(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na < 1e-15 or nb < 1e-15:
        return 0.0
    return np.dot(a, b) / (na * nb)

## Approximate Newton Direction via Conjugate Gradient

### Why CG-Newton Instead of Full Hessian?

For 32x32 with 4 layers, we have $n = 4096$ parameters. The full Hessian $H \in \mathbb{R}^{4096 \times 4096}$ has ~16.8 million entries. While technically feasible to store, computing it via finite differences requires $O(n)$ gradient evaluations, and inverting it requires $O(n^3)$ operations. This is prohibitively expensive when we need to do it at 10 checkpoints across 5 seeds.

### The CG-Newton Approach

Instead, we solve $H d = -g$ using the conjugate gradient (CG) algorithm, which only requires **Hessian-vector products** $Hv$, never the full Hessian. Each Hessian-vector product is computed via central finite differences:

$$Hv \approx \frac{\nabla \mathcal{L}(\theta + \epsilon v) - \nabla \mathcal{L}(\theta - \epsilon v)}{2\epsilon}$$

This requires just 2 gradient evaluations per CG iteration, regardless of parameter count. With 30 CG iterations, we get a good approximation to $d_N = -H^{-1}g$ using only 60 gradient evaluations instead of 4096.

### What We Measure

At each checkpoint, we compute:
- $d_N^{\text{Muon}}$: the Newton direction at Muon's current parameter point
- $d_N^{\text{NormSGD}}$: the Newton direction at NormSGD's current parameter point

Then we measure how well each optimizer's actual step direction aligns with its respective Newton direction:
- $\cos(\text{Muon\_step}, d_N^{\text{Muon}})$: Muon's alignment with Newton at its own point
- $\cos(\text{NormSGD\_step}, d_N^{\text{NormSGD}})$: NormSGD's alignment with Newton at its own point

The **cosine advantage** at each checkpoint is the difference: $\Delta\cos_t = \cos_{\text{Muon}} - \cos_{\text{NormSGD}}$.

In [ ]:
def hessian_vec_product(theta, X, Y, v, eps=1e-5):
    """Hessian-vector product via finite differences: H @ v ~ (grad(theta+eps*v) - grad(theta-eps*v)) / (2*eps)."""
    vn = np.linalg.norm(v)
    if vn < 1e-15:
        return np.zeros_like(v)
    tp = theta + eps * v
    tm = theta - eps * v
    gp, _ = grad_fn_flat(tp, X, Y)
    gm, _ = grad_fn_flat(tm, X, Y)
    return (gp - gm) / (2 * eps)

In [ ]:
def cg_newton(theta, X, Y, g, max_iters=50, tol=1e-6):
    """Conjugate gradient to approximately solve H @ d = -g."""
    b = -g
    d = np.zeros_like(g)
    r = b.copy()
    p = r.copy()
    rsold = np.dot(r, r)
    if rsold < tol ** 2:
        return d
    for _ in range(max_iters):
        Ap = hessian_vec_product(theta, X, Y, p)
        pAp = np.dot(p, Ap)
        if abs(pAp) < 1e-15:
            break
        alpha = rsold / pAp
        d = d + alpha * p
        r = r - alpha * Ap
        rsnew = np.dot(r, r)
        if rsnew < tol ** 2:
            break
        p = r + (rsnew / rsold) * p
        rsold = rsnew
    return d

## Training Functions: Muon vs. NormSGD

These two training loops are structurally identical except for the per-layer update rule:

**Muon:** $\text{update}_l = \text{NewtonSchulz}(\nabla_{W_l} \mathcal{L})$ -- projects gradient onto the nearest orthogonal matrix, equalizing all singular values to 1.

**NormSGD:** $\text{update}_l = \nabla_{W_l} \mathcal{L} / \|\nabla_{W_l} \mathcal{L}\|_F$ -- normalizes the gradient to unit Frobenius norm, preserving its direction but fixing its magnitude.

Both apply momentum identically: $m_l^{(t)} = 0.9 \cdot m_l^{(t-1)} + \text{update}_l$.

The crucial difference is that Muon's Newton-Schulz **rotates** the gradient (changing singular vectors' relative weighting) while NormSGD only **rescales** it (preserving direction). This rotation is what gives Muon better Newton alignment -- it redistributes gradient energy from high-curvature directions to low-curvature ones.

These trajectory functions are used only during the LR sweep. The main experiment uses `run_trajectory_with_cosine` below.

In [ ]:
def train_muon_trajectory(weights_init, X, Y, lr):
    """Train with Muon, return loss at every step."""
    weights = [W.copy() for W in weights_init]
    mom = [np.zeros_like(W) for W in weights]
    losses = []
    for step in range(NUM_STEPS):
        loss = compute_loss(weights, X, Y)
        losses.append(loss)
        if not np.isfinite(loss) or loss > 1e10:
            losses.extend([float('inf')] * (NUM_STEPS - step - 1))
            break
        grads = compute_gradients(weights, X, Y)
        for l in range(N_LAYERS):
            mom[l] = MOMENTUM * mom[l] + newton_schulz(grads[l])
            weights[l] = weights[l] - lr * mom[l]
    return losses, weights

In [ ]:
def train_normsgd_trajectory(weights_init, X, Y, lr):
    """Train with NormSGD, return loss at every step."""
    weights = [W.copy() for W in weights_init]
    mom = [np.zeros_like(W) for W in weights]
    losses = []
    for step in range(NUM_STEPS):
        loss = compute_loss(weights, X, Y)
        losses.append(loss)
        if not np.isfinite(loss) or loss > 1e10:
            losses.extend([float('inf')] * (NUM_STEPS - step - 1))
            break
        grads = compute_gradients(weights, X, Y)
        for l in range(N_LAYERS):
            nrm = np.linalg.norm(grads[l], 'fro')
            normed = grads[l] / max(nrm, 1e-15)
            mom[l] = MOMENTUM * mom[l] + normed
            weights[l] = weights[l] - lr * mom[l]
    return losses, weights

## Parallel Trajectory with Cosine Measurement

This is the core measurement function. It runs both optimizers **simultaneously** from the same initialization, so at every step we can compare their positions in parameter space.

### The Measurement Protocol in Detail

At each checkpoint step $t_k$ (every 50 steps):

1. **Compute gradients** at both optimizers' current positions $\theta_{\text{Muon}}^{(t_k)}$ and $\theta_{\text{NormSGD}}^{(t_k)}$.

2. **Solve for Newton direction** at each position via CG: $d_N = -H^{-1}g$. Note that each optimizer gets its own Newton direction because they are at different points in parameter space with different Hessians.

3. **Compute optimizer step directions:**
   - Muon: concatenate Newton-Schulz-transformed per-layer gradients
   - NormSGD: concatenate Frobenius-normalized per-layer gradients

4. **Measure cosine similarity** $\cos(-\text{step}, d_N)$ for each. The negation is because the step is in the descent direction while $d_N = -H^{-1}g$ is also a descent direction.

5. **Accumulate** the cosine advantage: $\text{cumul}(t_k) = \sum_{j \leq k} [\cos_{\text{Muon}}^{(j)} - \cos_{\text{NormSGD}}^{(j)}]$.

### Why Both Optimizers Get Their Own Newton Direction

A subtlety: we do NOT compare both optimizers against the same Newton direction. Instead, each is compared against the Newton direction at its own current point. This is the correct comparison because what matters for compounding is whether each optimizer is moving in the locally optimal direction from where it currently stands. The compounding hypothesis predicts that Muon's better alignment puts it at a point where the next Newton direction is also better aligned -- a self-reinforcing cycle.

In [ ]:
def run_trajectory_with_cosine(weights_init, X, Y, lr_muon, lr_norm):
    """
    Run Muon and NormSGD in parallel from same init.
    At every MEASURE_EVERY steps, compute cosine(step, Newton) for both.
    """
    w_muon = [W.copy() for W in weights_init]
    w_norm = [W.copy() for W in weights_init]
    mom_muon = [np.zeros_like(W) for W in weights_init]
    mom_norm = [np.zeros_like(W) for W in weights_init]

    cos_advantages = []  # per-checkpoint cosine advantage
    loss_muon_at_cp = {}
    loss_norm_at_cp = {}
    cumul_cos = 0.0
    cumul_at_cp = {}

    for step in range(NUM_STEPS):
        l_muon = compute_loss(w_muon, X, Y)
        l_norm = compute_loss(w_norm, X, Y)

        if not np.isfinite(l_muon) or l_muon > 1e10:
            l_muon = float('inf')
        if not np.isfinite(l_norm) or l_norm > 1e10:
            l_norm = float('inf')

        # Measure cosine at checkpoints
        if (step + 1) in CHECKPOINTS or step == 0:
            theta_muon = pack(w_muon)
            g_muon_flat, grads_muon = grad_fn_flat(theta_muon, X, Y)

            theta_norm = pack(w_norm)
            g_norm_flat, grads_norm = grad_fn_flat(theta_norm, X, Y)

            # Newton direction at Muon's point (using CG)
            d_newton_muon = cg_newton(theta_muon, X, Y, g_muon_flat, max_iters=30)
            # Newton direction at NormSGD's point
            d_newton_norm = cg_newton(theta_norm, X, Y, g_norm_flat, max_iters=30)

            # Muon step direction (at Muon's point)
            muon_step = pack([newton_schulz(G) for G in grads_muon])
            # NormSGD step direction (at NormSGD's point)
            norm_step = pack([G / max(np.linalg.norm(G, 'fro'), 1e-15) for G in grads_norm])

            # Cosine with Newton
            cos_muon_newton = cosine(-muon_step, d_newton_muon)
            cos_norm_newton = cosine(-norm_step, d_newton_norm)
            cos_adv = cos_muon_newton - cos_norm_newton
            cos_advantages.append(cos_adv)
            cumul_cos += cos_adv

        # Record at checkpoints
        if (step + 1) in CHECKPOINTS:
            loss_muon_at_cp[step + 1] = l_muon
            loss_norm_at_cp[step + 1] = l_norm
            cumul_at_cp[step + 1] = cumul_cos

        # Take Muon step
        if np.isfinite(l_muon) and l_muon < 1e10:
            grads_m = compute_gradients(w_muon, X, Y)
            for l in range(N_LAYERS):
                mom_muon[l] = MOMENTUM * mom_muon[l] + newton_schulz(grads_m[l])
                w_muon[l] = w_muon[l] - lr_muon * mom_muon[l]

        # Take NormSGD step
        if np.isfinite(l_norm) and l_norm < 1e10:
            grads_n = compute_gradients(w_norm, X, Y)
            for l in range(N_LAYERS):
                nrm = np.linalg.norm(grads_n[l], 'fro')
                normed = grads_n[l] / max(nrm, 1e-15)
                mom_norm[l] = MOMENTUM * mom_norm[l] + normed
                w_norm[l] = w_norm[l] - lr_norm * mom_norm[l]

    return cos_advantages, cumul_at_cp, loss_muon_at_cp, loss_norm_at_cp

## Learning Rate Sweep

Each optimizer gets its own optimal learning rate, determined by sweeping on the first seed and selecting the LR that achieves the lowest final loss. This is essential for a fair comparison: Muon and NormSGD have different effective step sizes due to their different normalization schemes, so using the same LR would confound the directional quality comparison.

The sweep is done once on seed 0, then the best LR is used for all seeds. This is a deliberate design choice -- we want to test whether Muon's directional advantage compounds, not whether it is more robust to LR choice (that is H18a's territory).

In [ ]:
def sweep_lr(train_fn, weights_init, X, Y, candidates):
    best_lr, best_loss = candidates[0], float('inf')
    for lr in candidates:
        losses, _ = train_fn([W.copy() for W in weights_init], X, Y, lr)
        final = losses[-1] if losses else float('inf')
        if np.isfinite(final) and final < best_loss:
            best_loss = final
            best_lr = lr
    return best_lr

## Main Experiment: Measuring Compounding Across Seeds

The main function orchestrates the full experiment:

1. **LR sweep** on the first seed to find optimal learning rates for both optimizers
2. **Parallel trajectory runs** across 5 seeds, collecting cosine advantages and loss ratios at each checkpoint
3. **Two regression fits** to determine whether cosine compounding or simple exponential divergence better explains the data
4. **Statistical comparison** of $R^2$ values to adjudicate between the two models

### The Two Competing Models

**Model A -- Cosine Compounding:** $\log(\text{loss\_ratio}) = \alpha \cdot \text{cumul\_cos\_advantage} + \beta$. This says the loss ratio is driven by the accumulated directional quality difference. If cosine advantage varies nonlinearly over training (e.g., grows or shrinks), Model A will capture that, while Model B (which uses only step count) cannot.

**Model B -- Simple Exponential:** $\log(\text{loss\_ratio}) = \gamma \cdot T + \delta$. This says the loss ratio grows at a constant exponential rate regardless of what the cosine advantage is doing. Model B is the null hypothesis -- it says time alone explains everything, and the cosine advantage is just a correlate.

If Model A has higher $R^2$, it means the cosine advantage carries **additional predictive information** beyond what step count provides. This would confirm that directional quality is the causal mechanism, not just a side effect of Muon being "better" in some generic sense.

In [ ]:
def main():
    seeds = [42 + i * 137 for i in range(NUM_SEEDS)]

    print("=" * 100)
    print("H20a: COSINE COMPOUNDING -- Does +0.004/step Compound to 19x?")
    print("=" * 100)
    print(f"Network: {N_LAYERS}-layer deep linear {DIM}x{DIM}")
    print(f"Steps: {NUM_STEPS}, Checkpoints: {CHECKPOINTS}")
    print(f"Seeds: {NUM_SEEDS}")
    print(f"Total parameters: {N_LAYERS * DIM * DIM}")
    print(f"Newton approximation: CG with up to 30 iterations")
    print()

    lr_muon_candidates = [0.05, 0.03, 0.02, 0.015, 0.01, 0.007, 0.005, 0.003, 0.002, 0.001]
    lr_norm_candidates = [0.1, 0.07, 0.05, 0.03, 0.02, 0.01, 0.005, 0.003, 0.001]

    # =========================================================================
    # PHASE 1: Learning Rate Sweep
    # =========================================================================
    print("=" * 80)
    print("PHASE 1: LEARNING RATE SWEEP")
    print("=" * 80)
    print("Sweeping LR on first seed to find optimal rates for both optimizers.")
    print(f"  Muon candidates:   {lr_muon_candidates}")
    print(f"  NormSGD candidates: {lr_norm_candidates}")
    print()

    X0, Y0 = make_data(seeds[0])
    w0 = init_weights(seeds[0] + 5000)

    print(f"  Data: X shape={X0.shape}, Y shape={Y0.shape}")
    print(f"  Initial loss: {compute_loss(w0, X0, Y0):.6f}")
    print()

    best_lr_muon = sweep_lr(train_muon_trajectory, w0, X0, Y0, lr_muon_candidates)
    best_lr_norm = sweep_lr(train_normsgd_trajectory, w0, X0, Y0, lr_norm_candidates)
    print(f"  Best Muon LR:   {best_lr_muon}")
    print(f"  Best NormSGD LR: {best_lr_norm}")
    print(f"  LR ratio (NormSGD/Muon): {best_lr_norm / best_lr_muon:.2f}x")
    print()
    print("  Note: Different optimal LRs are expected because Newton-Schulz and")
    print("  Frobenius normalization produce updates with different effective magnitudes.")
    print()

    # =========================================================================
    # PHASE 2: Main Measurement Across Seeds
    # =========================================================================
    print("=" * 80)
    print("PHASE 2: PARALLEL TRAJECTORY MEASUREMENT")
    print("=" * 80)
    print("Running Muon and NormSGD in parallel from same init for each seed.")
    print("At each checkpoint, computing CG-Newton direction and cosine alignment.")
    print()

    all_cumul = {cp: [] for cp in CHECKPOINTS}
    all_loss_ratio = {cp: [] for cp in CHECKPOINTS}
    all_cos_advs = []

    for si, seed in enumerate(seeds):
        print(f"  --- Seed {si + 1}/{NUM_SEEDS} (seed={seed}) ---")
        X, Y = make_data(seed)
        w_init = init_weights(seed + 5000)

        init_loss = compute_loss(w_init, X, Y)
        print(f"    Initial loss: {init_loss:.6f}")

        cos_advs, cumul_at_cp, loss_m_cp, loss_n_cp = run_trajectory_with_cosine(
            w_init, X, Y, best_lr_muon, best_lr_norm
        )
        all_cos_advs.append(cos_advs)

        # Print per-checkpoint detail for this seed
        print(f"    {'Step':>6}  {'Loss Muon':>12}  {'Loss NormSGD':>14}  {'Ratio':>8}  {'Cumul Cos':>12}  {'Cos Adv':>10}")
        for cp in CHECKPOINTS:
            if cp in cumul_at_cp and cp in loss_m_cp and cp in loss_n_cp:
                all_cumul[cp].append(cumul_at_cp[cp])
                lm = loss_m_cp[cp]
                ln = loss_n_cp[cp]
                if np.isfinite(lm) and np.isfinite(ln) and lm > 1e-30:
                    ratio = ln / lm
                else:
                    ratio = float('nan')
                all_loss_ratio[cp].append(ratio)

                # Find per-checkpoint cos advantage
                cp_idx = CHECKPOINTS.index(cp)
                cos_adv_val = cos_advs[cp_idx] if cp_idx < len(cos_advs) else float('nan')

                ratio_str = f"{ratio:.2f}x" if np.isfinite(ratio) else "nan"
                print(f"    {cp:>6}  {lm:>12.6f}  {ln:>14.6f}  {ratio_str:>8}  {cumul_at_cp[cp]:>12.4f}  {cos_adv_val:>+10.6f}")

        # Print per-seed summary
        final_cp = CHECKPOINTS[-1]
        if all_loss_ratio[final_cp]:
            final_ratio = all_loss_ratio[final_cp][-1]
            final_cumul = all_cumul[final_cp][-1]
            print(f"    SEED SUMMARY: final loss_ratio = {final_ratio:.2f}x, "
                  f"cumul_cos_advantage = {final_cumul:.4f}")
            if np.isfinite(final_ratio) and final_ratio > 1:
                print(f"    -> Muon is {final_ratio:.1f}x better than NormSGD at step {final_cp}")
            elif np.isfinite(final_ratio):
                print(f"    -> NormSGD is {1/final_ratio:.1f}x better than Muon at step {final_cp}")
        print()

    # =========================================================================
    # PHASE 3: AGGREGATION AND REGRESSION
    # =========================================================================
    print("=" * 80)
    print("PHASE 3: AGGREGATE RESULTS AND REGRESSION ANALYSIS")
    print("=" * 80)
    print()

    print("--- Checkpoint-Level Summary (averaged across seeds) ---")
    print(f"  {'Step':>6}  {'Mean Cumul Cos':>16}  {'Std':>8}  {'Mean Loss Ratio':>16}  {'Std':>10}  {'log(ratio)':>12}")
    print("  " + "-" * 80)

    cps_valid = []
    log_ratios = []
    cumuls = []
    steps_valid = []

    for cp in CHECKPOINTS:
        ratios = [r for r in all_loss_ratio[cp] if np.isfinite(r) and r > 0]
        cums = [c for c in all_cumul[cp] if np.isfinite(c)]
        if ratios and cums:
            mr = np.mean(ratios)
            sr = np.std(ratios)
            mc = np.mean(cums)
            sc = np.std(cums)
            lr_val = np.log(max(mr, 1e-30))
            print(f"  {cp:>6}  {mc:>16.4f}  {sc:>8.4f}  {mr:>15.4f}x  {sr:>10.4f}  {lr_val:>12.4f}")
            cps_valid.append(cp)
            log_ratios.append(lr_val)
            cumuls.append(mc)
            steps_valid.append(cp)

    print()
    print(f"  Valid checkpoints for regression: {len(cps_valid)}")
    if len(cps_valid) >= 2:
        print(f"  Loss ratio range: {np.exp(min(log_ratios)):.4f}x to {np.exp(max(log_ratios)):.4f}x")
        print(f"  Cumul cos range:  {min(cumuls):.4f} to {max(cumuls):.4f}")
    print()

    # ---- FIT A: log(loss_ratio) vs cumulative cosine advantage ----
    print("=" * 80)
    print("FIT A: log(loss_ratio) vs cumulative_cosine_advantage")
    print("=" * 80)
    print("  Model: log(loss_ratio) = alpha * cumul_cos + beta")
    print("  Hypothesis: if alpha > 0 and R^2 is high, each unit of cumulative")
    print("  cosine advantage multiplies the loss ratio by exp(alpha).")
    print()

    if len(cumuls) >= 3:
        slope_cos, intercept_cos = np.polyfit(cumuls, log_ratios, 1)
        # R^2
        pred_cos = np.array(cumuls) * slope_cos + intercept_cos
        ss_res_cos = np.sum((np.array(log_ratios) - pred_cos) ** 2)
        ss_tot = np.sum((np.array(log_ratios) - np.mean(log_ratios)) ** 2)
        r2_cos = 1 - ss_res_cos / max(ss_tot, 1e-30)

        print(f"  Fit: log(ratio) = {slope_cos:.4f} * cumul_cos + {intercept_cos:.4f}")
        print(f"  R^2 = {r2_cos:.4f}")
        print(f"  Slope interpretation: each +1.0 unit of cumulative cosine advantage")
        print(f"    multiplies the loss ratio by exp({slope_cos:.4f}) = {np.exp(slope_cos):.2f}x")
        print()
        print(f"  Residuals (predicted vs actual log-ratio):")
        for i, cp in enumerate(cps_valid):
            residual = log_ratios[i] - pred_cos[i]
            print(f"    Step {cp:>4}: actual={log_ratios[i]:>8.4f}, predicted={pred_cos[i]:>8.4f}, residual={residual:>+8.4f}")
    else:
        r2_cos = float('nan')
        print("  Not enough data points for fit (need >= 3)")
    print()

    # ---- FIT B: log(loss_ratio) vs step count ----
    print("=" * 80)
    print("FIT B: log(loss_ratio) vs step_count (null model)")
    print("=" * 80)
    print("  Model: log(loss_ratio) = gamma * T + delta")
    print("  This is the null hypothesis: loss ratio grows exponentially in time,")
    print("  and cosine advantage is merely correlated with time.")
    print()

    if len(steps_valid) >= 3:
        slope_t, intercept_t = np.polyfit(steps_valid, log_ratios, 1)
        pred_t = np.array(steps_valid) * slope_t + intercept_t
        ss_res_t = np.sum((np.array(log_ratios) - pred_t) ** 2)
        r2_t = 1 - ss_res_t / max(ss_tot, 1e-30)

        print(f"  Fit: log(ratio) = {slope_t:.6f} * T + {intercept_t:.4f}")
        print(f"  R^2 = {r2_t:.4f}")
        print(f"  Rate: {slope_t:.6f}/step")
        print(f"  Predicted ratio at T={NUM_STEPS}: exp({slope_t:.6f} * {NUM_STEPS}) = {np.exp(slope_t * NUM_STEPS):.2f}x")
        print()
        print(f"  Residuals (predicted vs actual log-ratio):")
        for i, cp in enumerate(cps_valid):
            residual = log_ratios[i] - pred_t[i]
            print(f"    Step {cp:>4}: actual={log_ratios[i]:>8.4f}, predicted={pred_t[i]:>8.4f}, residual={residual:>+8.4f}")
    else:
        r2_t = float('nan')
        print("  Not enough data points for fit (need >= 3)")
    print()

    # ---- KEY TEST: which fit is better? ----
    print("=" * 80)
    print("KEY TEST: Does cumulative cosine predict loss ratio BETTER than step count?")
    print("=" * 80)
    print()
    if np.isfinite(r2_cos) and np.isfinite(r2_t):
        delta_r2 = r2_cos - r2_t
        print(f"  R^2(cosine model A) = {r2_cos:.6f}")
        print(f"  R^2(step model B)   = {r2_t:.6f}")
        print(f"  Delta R^2 = {delta_r2:+.6f}")
        print()
        if r2_cos > r2_t + 0.01:
            print(f"  RESULT: COSINE COMPOUNDING CONFIRMED (R^2_A > R^2_B by {delta_r2:.4f})")
            print(f"  The cumulative cosine advantage explains the loss ratio BETTER than")
            print(f"  step count alone. This means the tiny per-step directional advantage")
            print(f"  is genuinely compounding through the curvature landscape, not just")
            print(f"  a side effect of time passing.")
        elif r2_t > r2_cos + 0.01:
            print(f"  RESULT: SIMPLE EXPONENTIAL DIVERGENCE (R^2_B > R^2_A by {-delta_r2:.4f})")
            print(f"  Step count alone explains the loss ratio better than cumulative cosine.")
            print(f"  The cosine advantage may be present but is not the primary driver --")
            print(f"  perhaps some other mechanism (e.g., conditioning improvement, spectral")
            print(f"  dynamics) is responsible for the exponential divergence.")
        else:
            print(f"  RESULT: INCONCLUSIVE (|Delta R^2| = {abs(delta_r2):.4f} < 0.01)")
            print(f"  Both models fit approximately equally well. The cosine advantage may")
            print(f"  grow linearly in time (making the two models equivalent), or there")
            print(f"  may be insufficient variation to distinguish them.")

        print()
        print(f"  Additional diagnostic: if cumul_cos grows linearly in T, the two models")
        print(f"  are algebraically equivalent and cannot be distinguished. Checking:")
        if len(cumuls) >= 3 and len(steps_valid) >= 3:
            slope_ct, intercept_ct = np.polyfit(steps_valid, cumuls, 1)
            pred_ct = np.array(steps_valid) * slope_ct + intercept_ct
            ss_res_ct = np.sum((np.array(cumuls) - pred_ct) ** 2)
            ss_tot_ct = np.sum((np.array(cumuls) - np.mean(cumuls)) ** 2)
            r2_ct = 1 - ss_res_ct / max(ss_tot_ct, 1e-30)
            print(f"    cumul_cos vs T: R^2 = {r2_ct:.4f}")
            if r2_ct > 0.99:
                print(f"    -> Cumul cos is nearly perfectly linear in T (R^2={r2_ct:.4f}).")
                print(f"       The two models are algebraically equivalent in this regime.")
                print(f"       To distinguish them, we would need conditions where cosine")
                print(f"       advantage varies nonlinearly (e.g., changing depth, LR schedule).")
            else:
                print(f"    -> Cumul cos is NOT linear in T (R^2={r2_ct:.4f}).")
                print(f"       The two models are distinguishable, so the R^2 comparison is valid.")
    else:
        print("  INCONCLUSIVE (insufficient data for one or both fits)")
    print()

    # ---- Per-step cosine advantage trend ----
    print("=" * 80)
    print("PER-STEP COSINE ADVANTAGE EVOLUTION")
    print("=" * 80)
    print("  How does the per-step advantage change over training?")
    print("  If it grows: accelerating compounding (super-exponential)")
    print("  If it shrinks: decelerating compounding (sub-exponential)")
    print("  If constant: steady compounding (pure exponential)")
    print()
    if all_cos_advs:
        n_cp = len(all_cos_advs[0])
        per_step_means = []
        for ci in range(min(n_cp, len(CHECKPOINTS))):
            vals = [s[ci] for s in all_cos_advs if ci < len(s)]
            cp_step = CHECKPOINTS[ci] if ci < len(CHECKPOINTS) else ci * MEASURE_EVERY
            mean_val = np.mean(vals)
            std_val = np.std(vals)
            per_step_means.append(mean_val)
            sign = "MUON LEADS" if mean_val > 0 else "NORMSGD LEADS"
            print(f"    Step ~{cp_step:>4}: mean cos advantage = {mean_val:+.6f}  "
                  f"(std={std_val:.6f}) [{sign}]")

        # Check trend
        if len(per_step_means) >= 3:
            early_mean = np.mean(per_step_means[:3])
            late_mean = np.mean(per_step_means[-3:])
            print()
            print(f"    Early (first 3 checkpoints) mean: {early_mean:+.6f}")
            print(f"    Late  (last 3 checkpoints)  mean: {late_mean:+.6f}")
            if abs(late_mean) > abs(early_mean) * 1.5:
                print(f"    -> Advantage is GROWING over training (accelerating compounding)")
            elif abs(late_mean) < abs(early_mean) * 0.5:
                print(f"    -> Advantage is SHRINKING over training (decelerating compounding)")
            else:
                print(f"    -> Advantage is roughly STABLE over training (steady compounding)")

    print(f"\n{'=' * 100}")
    print("EXPERIMENT COMPLETE")
    print(f"{'=' * 100}")

In [ ]:
if __name__ == '__main__':
    main()

## Conclusions and Interpretation

### What This Experiment Tests

H20a addresses the most important quantitative puzzle in the Muon research program: how does a +0.004 per-step cosine advantage produce a 19x final loss ratio? The two candidate explanations are:

1. **Geometric compounding via curvature coupling** (Model A): The per-step directional advantage feeds back through the loss landscape. Better direction at step $t$ leads to a better region at step $t+1$, which leads to a better gradient, which leads to a better direction. The loss ratio grows as $\exp(\alpha \cdot \text{cumul\_cos})$.

2. **Simple exponential divergence** (Model B): The loss ratio grows exponentially in time for some generic reason (e.g., the optimizers' trajectories diverge in parameter space at a constant rate). The cosine advantage happens to be correlated with time but is not the causal driver.

### How to Read the Results

- If **R^2(A) > R^2(B)**: The cosine advantage carries predictive information beyond step count. This strongly supports the compounding mechanism as causal.
- If **R^2(B) > R^2(A)**: Step count is a better predictor. The cosine advantage is epiphenomenal -- it correlates with the outcome but does not drive it.
- If **R^2(A) ~ R^2(B)**: The cumulative cosine likely grows linearly in time, making the two models algebraically equivalent. The experiment is inconclusive and needs conditions where cosine advantage varies nonlinearly (H20b tests this via depth/curvature variation).

### The Identifiability Problem

There is an inherent difficulty: if the per-step cosine advantage is approximately constant over training, then cumulative cosine is proportional to step count, and the two models are identical. This is actually the *expected* scenario for a fixed architecture at steady state. The experiment is most informative when the cosine advantage varies over training -- either growing (as curvature changes) or shrinking (as both optimizers approach convergence). The per-step evolution section of the output diagnoses this.

### Connection to the Broader Research Program

- **H15a** established that the per-step advantage exists but is small (+0.004).
- **H3** established that the final outcome difference is large (19x).
- **H20a** (this experiment) tests whether compounding bridges the gap.
- **H20b** will test whether the compounding rate depends on curvature (depth, conditioning), which would further confirm the curvature-coupling mechanism.

If compounding is confirmed, it completes a causal chain: Newton-Schulz orthogonalization $\to$ better per-step Newton alignment $\to$ geometric compounding via curvature-landscape coupling $\to$ large final loss advantage. This is the core mechanism claim of the "Muon as RG gauge fixing" framework.